In [1]:
import pymupdf
from torch.utils.data import DataLoader , Dataset
from datasets import Dataset , load_dataset

d:\interview_llm\.interview_llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset=load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K")

Generating train split: 100%|██████████| 75197/75197 [00:01<00:00, 59524.75 examples/s]


In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['lang', 'raw_index', 'index', 'seed', 'openai_fingerprint', 'problem', 'solution'],
        num_rows: 75197
    })
})

In [31]:
import numpy as np

sample = dataset["train"].select(range(75197))  # sampling is enough, full 75K is slow & unnecessary
lengths = []

for p, s in zip(sample["problem"], sample["solution"]):
    text = f"Problem: {p}, Solution: {s}"
    length = len(tokenizer(text)["input_ids"])  # token count, not character count
    lengths.append(length)

lengths = np.array(lengths)
print("mean:", lengths.mean())
print("median:", np.median(lengths))
print("90th pct:", np.percentile(lengths, 90))
print("95th pct:", np.percentile(lengths, 95))
print("max:", lengths.max())


mean: 503.73920502147695
median: 485.0
90th pct: 699.0
95th pct: 780.0
max: 5529


In [32]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.is_bf16_supported())

NVIDIA GeForce GTX 1650
True


In [33]:
sample_tokens = tokenizer("hello" + tokenizer.eos_token)["input_ids"]
print(sample_tokens[-1] == tokenizer.eos_token_id)   # should print True

True


In [6]:
from transformers import AutoTokenizer
tokenizer= AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-0.5B-Instruct")

In [ ]:
def tokenize(example) : 
  instruction = example["problem"]  
  solution   =  example["solution"]
  text = f"Problem :  {instruction}  ,  Solution : {solution}"
  tokens = tokenizer(text = text , truncation=True , padding="max_length" , max_length=2000)
  tokens["labels"]  = tokens["input_ids"].copy()
  return tokens
  

In [23]:
tokenize(dataset['train'][0])

{'input_ids': [31198, 549, 220, 4319, 2610, 525, 2661, 1378, 18386, 11, 362, 323, 425, 11, 1817, 315, 3084, 308, 13, 1446, 1184, 311, 2736, 264, 55712, 5666, 389, 1493, 18386, 323, 2550, 279, 12942, 1334, 7110, 77, 1699, 785, 55712, 315, 1378, 18386, 362, 323, 425, 374, 4512, 438, 11017, 7190, 77, 12, 6771, 356, 387, 279, 12942, 1334, 315, 3084, 220, 17, 77, 12, 16, 11, 1380, 356, 989, 60, 284, 7851, 96, 4346, 3809, 60, 353, 425, 989, 13333, 2467, 369, 502, 284, 1932, 7, 15, 11, 600, 5279, 10, 16, 8, 311, 1308, 1956, 11, 308, 12, 16, 72341, 77, 1699, 7985, 264, 729, 476, 1714, 311, 2736, 279, 55712, 5666, 323, 470, 279, 12942, 1334, 356, 7110, 77, 1699, 5152, 32232, 25, 1124, 77, 73594, 10821, 1699, 3215, 4159, 29, 55712, 19066, 4159, 29, 264, 11, 4621, 4159, 29, 293, 10699, 77, 13874, 61069, 77, 1699, 2505, 7190, 77, 12, 9043, 18386, 264, 323, 293, 315, 3084, 308, 320, 16, 2651, 308, 2651, 220, 16, 15, 61, 20, 701, 1380, 1817, 2392, 315, 279, 1334, 374, 458, 7546, 10293, 16, 15, 61, 2

In [30]:
tokenize_dataset  =dataset.map(tokenize,batched=False ,remove_columns=dataset["train"].column_names)

Map: 100%|██████████| 75197/75197 [05:14<00:00, 239.31 examples/s]


In [29]:
tokenize_dataset

DatasetDict({
    train: Dataset({
        features: ['lang', 'raw_index', 'index', 'seed', 'openai_fingerprint', 'problem', 'solution', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 75197
    })
})

In [2]:
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()
HF_KEY = os.getenv("HF_ACCESS_KEY")

In [5]:
def load_data(path)  :
  text = []
  pdf = pymupdf.open(path)
  for page in pdf : 
    text.append(page.get_text())
  pdf.close()
  return text

In [6]:
text = load_data("Aditya_Pandey_Honeywell_DS_CoverLetter.pdf")

In [7]:
text

["Aditya Pandey\npandeyaditya2110@gmail.com  ·  +91 82738 37221  ·  Dehradun, India  ·  LinkedIn  ·  GitHub\n4 June 2026\nHiring Team\nHoneywell — Data Scientist I\nRe: Data Scientist I — Application by Aditya Pandey\nDear Hiring Team,\nI am writing to apply for the Data Scientist I position at Honeywell. As an M.Sc. Computer Science\ngraduate with end-to-end experience building and deploying machine learning systems, I am excited by\nthe opportunity to apply data-driven thinking to Honeywell's challenges in automation, industrial AI, and\nenergy transition — domains where the stakes of getting the model right genuinely matter.\nMy practical experience aligns directly with the core responsibilities of this role. I have built and delivered\nmultiple end-to-end data science projects in Python — from data ingestion and feature engineering\nthrough model training, evaluation, and production deployment. A strong example is my Network Intrusion\nDetection System, trained on the UNSW-NB15 dat

In [8]:
from transformers import AutoTokenizer

In [9]:
model_nmae = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_nmae)
chunk_size = 512

In [11]:
tokenizer.pad_token = tokenizer.eos_token

In [12]:
def chunking(text) :
  text_string = text[0]
  list1 = text_string.split()
  chunks = []
  chunk = []
  for word in list1 :
    chunk.append(word)
    if(len(chunk)>=30) :
      chunks.append(" ".join(chunk))
      chunk = []
  if(chunk != []) :
    chunks.append(" ".join(chunk))
    

  return chunks

    
    



In [13]:
chunks = chunking(text)

In [14]:
data = []
for chunk in chunks :
  d1 = {"text"  : chunk}
  data.append(d1)

In [15]:
data

[{'text': 'Aditya Pandey pandeyaditya2110@gmail.com · +91 82738 37221 · Dehradun, India · LinkedIn · GitHub 4 June 2026 Hiring Team Honeywell — Data Scientist I Re: Data Scientist I — Application'},
 {'text': 'by Aditya Pandey Dear Hiring Team, I am writing to apply for the Data Scientist I position at Honeywell. As an M.Sc. Computer Science graduate with end-to-end experience building and'},
 {'text': "deploying machine learning systems, I am excited by the opportunity to apply data-driven thinking to Honeywell's challenges in automation, industrial AI, and energy transition — domains where the stakes of"},
 {'text': 'getting the model right genuinely matter. My practical experience aligns directly with the core responsibilities of this role. I have built and delivered multiple end-to-end data science projects in Python'},
 {'text': '— from data ingestion and feature engineering through model training, evaluation, and production deployment. A strong example is my Network Intrusion Det

In [16]:
dataset = Dataset.from_list(data)

In [17]:
dataset

Dataset({
    features: ['text'],
    num_rows: 14
})

In [18]:
dataset["text"]

Column(['Aditya Pandey pandeyaditya2110@gmail.com · +91 82738 37221 · Dehradun, India · LinkedIn · GitHub 4 June 2026 Hiring Team Honeywell — Data Scientist I Re: Data Scientist I — Application', 'by Aditya Pandey Dear Hiring Team, I am writing to apply for the Data Scientist I position at Honeywell. As an M.Sc. Computer Science graduate with end-to-end experience building and', "deploying machine learning systems, I am excited by the opportunity to apply data-driven thinking to Honeywell's challenges in automation, industrial AI, and energy transition — domains where the stakes of", 'getting the model right genuinely matter. My practical experience aligns directly with the core responsibilities of this role. I have built and delivered multiple end-to-end data science projects in Python', '— from data ingestion and feature engineering through model training, evaluation, and production deployment. A strong example is my Network Intrusion Detection System, trained on the UNSW-NB15 datase

In [ ]:
def tokenize(dataset) :
  tokens = tokenizer(dataset["text"] , truncation=True , padding="max_length" , max_length=64)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [20]:
tokens = []
for data in dataset :
  tokens.append(tokenize(data))

In [21]:
tokens

[{'input_ids': [1, 2087, 537, 29874, 349, 4182, 29891, 282, 4182, 29891, 328, 537, 29874, 29906, 29896, 29896, 29900, 29992, 21980, 29889, 510, 2880, 718, 29929, 29896, 29871, 29947, 29906, 29955, 29941, 29947, 29871, 29941, 29955, 29906, 29906, 29896, 2880, 897, 1092, 328, 348, 29892, 7513, 2880, 28547, 797, 2880, 25492, 29871, 29946, 5306, 29871, 29906, 29900, 29906, 29953, 379, 8491, 8583, 379, 4992, 5872, 813], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [1, 2087, 537, 29874, 349, 4182, 29891, 282, 4182, 29891, 328, 537, 29874, 29906, 29896, 29896, 29900, 29992, 21980, 29889, 510, 2880, 718, 29929, 29896, 29871, 29947, 29906, 29955, 29941, 29947, 29871, 29941, 29955, 29906, 29906, 29896, 2880, 897, 1092, 328, 348, 29892, 7513, 2880, 28547, 797, 2880, 25492, 29871, 29946, 5306, 29871, 29906, 29900, 29906, 29

In [22]:
tokenize_dataset = dataset.map(tokenize , batched=True , remove_columns="text")

Map: 100%|██████████| 14/14 [00:00<00:00, 350.00 examples/s]


In [23]:
tokenize_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 14
})

In [24]:
from transformers import AutoModelForCausalLM

In [30]:
model = AutoModelForCausalLM.from_pretrained(model_nmae)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 832.80it/s]


In [2]:
from transformers import TrainingArguments , Trainer
from peft import LoraConfig , get_peft_model , TaskType
from datasets import load_dataset

In [3]:
import bitsandbytes , accelerate  
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
from datasets import load_dataset 
from transformers import Trainer , TrainingArguments , BitsAndBytesConfig 
import bitsandbytes 
from peft import get_peft_model  , LoraConfig , TaskType ,prepare_model_for_kbit_training
import torch



W0619 15:58:13.460000 17568 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [32]:
help(TrainingArguments)

Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing

In [34]:
train_args = TrainingArguments(
  output_dir="./llama-lora" ,
   num_train_epochs=2 , per_device_train_batch_size=2 , save_steps=500 ,save_total_limit=2 , logging_steps=50,learning_rate=2e-5
  ,fp16=True , report_to="none"
)

In [35]:
trainer  = Trainer(model ,train_dataset=tokenize_dataset , args=train_args)

In [37]:
lora_config = LoraConfig(task_type=TaskType.CAUSAL_LM ,r= 8  , lora_alpha = 16 , target_modules = ["q_proj" , "v_proj"],lora_dropout=0.05 , bias = "none")

In [38]:
lora_model = get_peft_model(model , lora_config)

W0618 14:36:43.525000 14524 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [42]:
trainer = Trainer(model = lora_model , args= train_args , train_dataset=tokenize_dataset)

In [43]:
trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 22.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 6.98 GiB is allocated by PyTorch, and 44.85 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [4]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-0.5B-Instruct")

In [5]:
tokenizer.SPECIAL_TOKENS_ATTRIBUTES

['bos_token',
 'eos_token',
 'unk_token',
 'sep_token',
 'pad_token',
 'cls_token',
 'mask_token']

In [7]:

print(tokenizer.special_tokens_map)

{'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}


In [9]:
# 1. Is it a known token in the vocab at all?
print(tokenizer.convert_tokens_to_ids("<|im_start|>"))
# if it returns a real integer (not None/unk_id), it exists as a token

# 2. Does the tokenizer have a chat template that uses it?

151644


In [10]:
print(tokenizer.chat_template)



{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [14]:
# 3. What does applying that template actually produce?
print(tokenizer.apply_chat_template(
    [{"role": "user", "content": "hello"} ],
    tokenize=False,
    add_generation_prompt=True
))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
hello<|im_end|>
<|im_start|>assistant

